<a href="https://colab.research.google.com/github/Naveed1503/Solar-Energy/blob/main/SolarEnergy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --------------------------------------------
# SOLAR ENERGY PREDICTION USING BIG DATA (PySpark)
# --------------------------------------------

!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Start Spark session
spark = SparkSession.builder.appName("SolarEnergyPrediction").getOrCreate()

# Load dataset
df = spark.read.csv("/content/Plant_1_Generation_Data.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5)

# Select relevant features
data = df.select(
    "DC_POWER",
    "AC_POWER"
)

# Remove nulls
data = data.na.drop()

# Feature Engineering
assembler = VectorAssembler(
    inputCols=["DC_POWER"],
    outputCol="features"
)

final = assembler.transform(data).select("features", "AC_POWER")

# Train-test split
train, test = final.randomSplit([0.8, 0.2], seed=42)

# Model (Random Forest)
model = RandomForestRegressor(featuresCol="features", labelCol="AC_POWER")
rf_model = model.fit(train)

# Predictions
predictions = rf_model.transform(test)
predictions.show(5)

# Evaluation
evaluator = RegressionEvaluator(
    labelCol="AC_POWER",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)
r2 = evaluator.setMetricName("r2").evaluate(predictions)

print("RMSE :", rmse)
print("R2 Score :", r2)


root
 |-- DATE_TIME: string (nullable = true)
 |-- PLANT_ID: integer (nullable = true)
 |-- SOURCE_KEY: string (nullable = true)
 |-- DC_POWER: double (nullable = true)
 |-- AC_POWER: double (nullable = true)
 |-- DAILY_YIELD: double (nullable = true)
 |-- TOTAL_YIELD: double (nullable = true)

+----------------+--------+---------------+--------+--------+-----------+-----------+
|       DATE_TIME|PLANT_ID|     SOURCE_KEY|DC_POWER|AC_POWER|DAILY_YIELD|TOTAL_YIELD|
+----------------+--------+---------------+--------+--------+-----------+-----------+
|15-05-2020 00:00| 4135001|1BY6WEcLGh8j5v7|     0.0|     0.0|        0.0|  6259559.0|
|15-05-2020 00:00| 4135001|1IF53ai7Xc0U56Y|     0.0|     0.0|        0.0|  6183645.0|
|15-05-2020 00:00| 4135001|3PZuoBAID5Wc2HD|     0.0|     0.0|        0.0|  6987759.0|
|15-05-2020 00:00| 4135001|7JYdWkrLSPkdwr4|     0.0|     0.0|        0.0|  7602960.0|
|15-05-2020 00:00| 4135001|McdE0feGgRqW7Ca|     0.0|     0.0|        0.0|  7158964.0|
+---------------

In [ ]:
!pip install pyspark streamlit pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 45.3 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------
st.set_page_config(page_title="Solar Energy Prediction", layout="wide", page_icon="🌞")

st.markdown("""
<style>
.big-title {
    font-size: 40px !important;
    font-weight: 800 !important;
    background: linear-gradient(90deg,#ff8a00,#ffc000);
    -webkit-background-clip: text;
    color: transparent;
}
.card {
    padding: 18px;
    border-radius: 18px;
    background: rgba(255,255,255,0.85);
    box-shadow: 0 8px 18px rgba(0,0,0,0.08);
    text-align:center;
    margin-bottom: 12px;
}
</style>
""", unsafe_allow_html=True)

st.markdown("<div class='big-title'>⚡ Solar Energy Forecasting Dashboard</div>", unsafe_allow_html=True)


# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("🧭 Navigation")

page = st.sidebar.radio(
    "Go to:",
    ["Prediction Panel", "Visualizations", "About Project"],
    index=0
)

page = page.strip().lower()

model_choice = st.sidebar.selectbox("Choose ML Model", ["Random Forest", "Gradient Boosting"])

# --------------------------------------------------
# SYNTHETIC DATA
# --------------------------------------------------
np.random.seed(42)
n = 900

df = pd.DataFrame({
    "IRRADIATION": np.random.uniform(50, 1100, n),
    "AMBIENT_TEMPERATURE": np.random.uniform(18, 42, n),
    "MODULE_TEMPERATURE": np.random.uniform(25, 80, n),
    "DC_POWER": np.random.uniform(100, 4000, n)
})

df["AC_POWER"] = (
    df["IRRADIATION"] * 0.45 +
    df["DC_POWER"] * 0.40 -
    (df["MODULE_TEMPERATURE"] - 30) * 1.8 +
    np.random.normal(0, 30, n)
)

X = df[["IRRADIATION", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "DC_POWER"]]
y = df["AC_POWER"]

# --------------------------------------------------
# TRAIN MODELS (FAST)
# --------------------------------------------------
rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42)
rf.fit(X, y)

gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05)
gb.fit(X, y)

def get_model():
    return rf if model_choice == "Random Forest" else gb

# --------------------------------------------------
# PAGE 1 — PREDICTION PANEL
# --------------------------------------------------
if page == "prediction panel":

    st.subheader("Live Solar Output Prediction")

    col1, col2 = st.columns(2)

    with col1:
        irr = st.slider("Solar Irradiation (W/m²)", 50, 1200, 850)
        amb = st.slider("Ambient Temperature (°C)", 10, 50, 32)

    with col2:
        mod = st.slider("Module Temperature (°C)", 25, 90, 50)
        dc = st.slider("DC Power (W)", 100, 5000, 2300)

    input_data = pd.DataFrame([[irr, amb, mod, dc]], columns=X.columns)

    model = get_model()
    prediction = model.predict(input_data)[0]

    efficiency = prediction / dc * 100
    level = "High" if prediction > 2500 else "Medium" if prediction > 1200 else "Low"

    st.markdown("### 📌 Prediction Summary")

    c1, c2, c3 = st.columns(3)

    c1.markdown(f"<div class='card'><h4>Predicted AC Power</h4><h2>{prediction:.1f} W</h2></div>", unsafe_allow_html=True)
    c2.markdown(f"<div class='card'><h4>Efficiency</h4><h2>{efficiency:.1f}%</h2></div>", unsafe_allow_html=True)
    c3.markdown(f"<div class='card'><h4>Generation Level</h4><h2>{level}</h2></div>", unsafe_allow_html=True)

    # GRAPH 1 – Irradiation
    irr_range = np.linspace(50, 1200, 80)
    df_test = pd.DataFrame({
        "IRRADIATION": irr_range,
        "AMBIENT_TEMPERATURE": amb,
        "MODULE_TEMPERATURE": mod,
        "DC_POWER": dc
    })
    pred_line = model.predict(df_test)

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=irr_range, y=pred_line, mode="lines"))
    fig1.add_trace(go.Scatter(x=[irr], y=[prediction], mode="markers", marker=dict(size=12, color="red")))
    fig1.update_layout(title="Effect of Irradiation on AC Power")
    st.plotly_chart(fig1, use_container_width=True)

    # FEATURE IMPORTANCE
    st.subheader("🏗 Feature Importance")

    importances = model.feature_importances_ if model_choice == "Random Forest" else [0.4,0.1,0.2,0.3]

    fig3 = px.bar(
        x=X.columns,
        y=importances,
        title="Feature Contribution",
        labels={"x": "Feature", "y": "Importance"}
    )
    st.plotly_chart(fig3, use_container_width=True)

# --------------------------------------------------
# PAGE 2 — VISUALIZATIONS
# --------------------------------------------------
elif page == "visualizations":

    st.subheader("📊 Training Data Visualization")

    fig_a = px.scatter(df, x="IRRADIATION", y="AC_POWER", opacity=0.5)
    st.plotly_chart(fig_a, use_container_width=True)

    fig_b = px.scatter(df, x="DC_POWER", y="AC_POWER", opacity=0.5)
    st.plotly_chart(fig_b, use_container_width=True)

# --------------------------------------------------
# PAGE 3 — ABOUT
# --------------------------------------------------
else:
    st.subheader("ℹ️ About This Project")
    st.write("""
This is a fully functional Solar Energy Forecasting System built using **Streamlit + Scikit-learn**
to avoid PySpark JVM issues inside Google Colab.

### Features:
- Manual input prediction
- Graphical visual explanation
- Fast ML models
- Dashboard UI
""")


Writing app.py


In [ ]:
NGROK_AUTH_TOKEN = "api key"  # ⬅️ Replace this actual token visit ngrok api website

In [ ]:
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Set auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Create tunnel for port 8501
public_url = ngrok.connect(8501)
print("✅ Streamlit Public URL:", public_url)

# Run Streamlit (in background)
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &>/dev/null&


✅ Streamlit Public URL: NgrokTunnel: "https://nonresistive-packly-jayde.ngrok-free.dev" -> "http://localhost:8501"
